# Qwen3.5-4B 파인튜닝 (Colab Pro A100)

**삼선뉴스 프로젝트** — 번역 태스크 파인튜닝

- 모델: `Qwen/Qwen3.5-4B` (Instruct)
- GPU: Colab Pro A100 (40GB) 기준
- 학습 방식: LoRA (Unsloth)
- 학습 데이터: `trainset_chat.jsonl` (prepare_finetune.py로 미리 변환된 파일)

---
**실행 전 체크리스트**
1. 런타임 → 런타임 유형 변경 → A100 GPU 선택
2. Google Drive에 `trainset_chat.jsonl` 업로드
3. 아래 `⚙️ CONFIG` 셀에서 경로 확인

## 1. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 패키지 설치

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv

if importlib.util.find_spec('torch') is None or 'COLAB_' in ''.join(os.environ.keys()):
    !uv pip install -qqq \
        'torch==2.8.0' 'triton>=3.3.0' numpy \
        'unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo' \
        'unsloth[base] @ git+https://github.com/unslothai/unsloth'
elif importlib.util.find_spec('unsloth') is None:
    !uv pip install -qqq unsloth

!uv pip install --upgrade --no-deps tokenizers trl==0.15.2 unsloth unsloth_zoo
!uv pip install -qqq datasets

## 3. ⚙️ CONFIG (여기서 설정 변경)

In [ ]:
# ────────────────────────────────────────────────
# 태스크 선택: 'translate' 또는 'summarize'
# ────────────────────────────────────────────────
TASK = "translate"  # 'translate' | 'summarize'

# ────────────────────────────────────────────────
# 모델
# ────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen3.5-4B"

# ────────────────────────────────────────────────
# Google Drive 데이터 경로
# Drive에 업로드할 파일:
#   trainset_translate_1000_chat.jsonl  (번역, 1000개)
#   trainset_summarize_1116_chat.jsonl  (요약, 1116개)
# ────────────────────────────────────────────────
JSONL_PATHS = {
    "translate": "/content/drive/MyDrive/samseon/trainset_translate_1000_chat.jsonl",
    "summarize": "/content/drive/MyDrive/samseon/trainset_summarize_1116_chat.jsonl",
}
TRAIN_JSONL_PATH = JSONL_PATHS[TASK]

# ────────────────────────────────────────────────
# 모델 저장 경로 (TASK별로 분리)
# ────────────────────────────────────────────────
OUTPUT_DIR     = f"/content/drive/MyDrive/samseon/models/finetune_output_{TASK}"
LORA_SAVE_PATH = f"/content/drive/MyDrive/samseon/models/qwen3_5_lora_{TASK}"

# ────────────────────────────────────────────────
# 하이퍼파라미터 (A100 40GB 기준)
# ────────────────────────────────────────────────
MAX_SEQ_LENGTH              = 512
LORA_R                      = 16
LORA_ALPHA                  = 16   # Unsloth 권장: lora_alpha == r
LORA_DROPOUT                = 0    # Unsloth 권장: dropout=0
PER_DEVICE_TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4    # 유효 배치 = 16
NUM_TRAIN_EPOCHS            = 3
LEARNING_RATE               = 1e-4
WARMUP_RATIO                = 0.05
SEED                        = 3407

print(f"TASK:  {TASK}")
print(f"JSONL: {TRAIN_JSONL_PATH}")
print(f"저장:  {LORA_SAVE_PATH}")

## 4. GPU 확인

In [ ]:
import torch

gpu = torch.cuda.get_device_properties(0)
total_gb = gpu.total_memory / 1e9
print(f"GPU: {gpu.name}")
print(f"VRAM: {total_gb:.1f} GB")
print(f"bf16 지원: {torch.cuda.is_bf16_supported()}")

if total_gb < 20:
    print("\n⚠️  VRAM 부족 — A100 또는 L4 런타임을 선택하세요.")
else:
    print("\n✅ A100 / L4 확인 — 학습 진행 가능")

## 5. 모델 로드

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_ID,
    max_seq_length= MAX_SEQ_LENGTH,
    load_in_4bit  = False,    # QLoRA 비활성화 (Unsloth 공식 권고: Qwen3.5는 QLoRA 비권장)
    load_in_16bit = True,     # bf16 LoRA
)

tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"모델 로드 완료: {MODEL_ID}")
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## 6. LoRA 어댑터 설정

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = SEED,
    max_seq_length             = MAX_SEQ_LENGTH,
)

model.print_trainable_parameters()

## 7. 데이터 로드 (trainset_chat.jsonl)

In [ ]:
import json
from datasets import Dataset

# prepare_finetune.py로 이미 변환된 JSONL 로드
with open(TRAIN_JSONL_PATH, encoding="utf-8") as f:
    train_data = [json.loads(line) for line in f if line.strip()]

print(f"학습 데이터: {len(train_data)}개")
print(f"\n[샘플 확인]")
sample = train_data[0]["messages"]
print(f"system:    {sample[0]['content'][:80]}...")
print(f"user:      {sample[1]['content'][:80]}...")
print(f"assistant: {sample[2]['content'][:80]}...")

## 8. Chat Template 적용 → Dataset 생성

In [ ]:
# messages → 토크나이저 chat template 적용
formatted = [{
    "text": tokenizer.apply_chat_template(
        item["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
} for item in train_data]

dataset = Dataset.from_list(formatted)

print(f"Dataset 크기: {len(dataset)}개")
print(f"\n[포맷 적용 샘플]\n{dataset[0]['text'][:400]}")

## 9. 학습

In [ ]:
import os
from trl import SFTTrainer, SFTConfig

os.makedirs(OUTPUT_DIR, exist_ok=True)

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model             = model,
    processing_class  = tokenizer,
    train_dataset     = dataset,
    args = SFTConfig(
        dataset_text_field          = "text",
        max_seq_length              = MAX_SEQ_LENGTH,
        per_device_train_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
        warmup_ratio                = WARMUP_RATIO,
        num_train_epochs            = NUM_TRAIN_EPOCHS,
        learning_rate               = LEARNING_RATE,
        bf16                        = torch.cuda.is_bf16_supported(),
        fp16                        = not torch.cuda.is_bf16_supported(),
        logging_steps               = 50,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "cosine",
        seed                        = SEED,
        output_dir                  = OUTPUT_DIR,
        save_steps                  = 50,
        save_total_limit            = 3,
        report_to                   = "none",
    ),
)

In [ ]:
start_mem = torch.cuda.memory_allocated() / 1e9
print(f"학습 전 GPU 메모리: {start_mem:.2f} GB")
print(f"유효 배치: {PER_DEVICE_TRAIN_BATCH_SIZE} x {GRADIENT_ACCUMULATION_STEPS} = {PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")

trainer_stats = trainer.train()

## 10. 학습 결과 확인

In [ ]:
runtime_sec    = trainer_stats.metrics.get("train_runtime", 0)
train_loss     = trainer_stats.metrics.get("train_loss", 0)
peak_mem       = torch.cuda.max_memory_allocated() / 1e9
total_vram     = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"학습 소요: {runtime_sec:.0f}초 ({runtime_sec/60:.1f}분)")
print(f"Train Loss: {train_loss:.4f}")
print(f"최대 VRAM: {peak_mem:.2f} GB / {total_vram:.1f} GB ({100*peak_mem/total_vram:.1f}%)")

## 11. 추론 테스트

In [ ]:
FastLanguageModel.for_inference(model)

# 학습 데이터 첫 번째 샘플로 테스트
sample_messages = train_data[0]["messages"]
test_messages   = sample_messages[:-1]   # system + user (assistant 제외)

text   = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens = 512,
        temperature    = 0.3,
        top_p          = 0.9,
        do_sample      = True,
        pad_token_id   = tokenizer.eos_token_id,
    )

generated = outputs[0][inputs["input_ids"].shape[1]:]
result    = tokenizer.decode(generated, skip_special_tokens=True)

print(f"[영문 입력]\n{sample_messages[1]['content'][:300]}")
print(f"\n[정답 번역]\n{sample_messages[2]['content'][:300]}")
print(f"\n[모델 출력]\n{result[:300]}")

## 12. 모델 저장

In [ ]:
os.makedirs(LORA_SAVE_PATH, exist_ok=True)

# LoRA 어댑터 저장 (기본)
model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)
print(f"LoRA 저장 완료: {LORA_SAVE_PATH}")

In [ ]:
# (선택) 병합 모델 16bit 저장 — Ollama/vLLM 배포용
MERGED_SAVE_PATH = "/content/drive/MyDrive/samseon/models/qwen3_5_merged_16bit"

if False:  # True로 변경하면 실행
    os.makedirs(MERGED_SAVE_PATH, exist_ok=True)
    model.save_pretrained_merged(MERGED_SAVE_PATH, tokenizer, save_method="merged_16bit")
    print(f"병합 모델 저장 완료: {MERGED_SAVE_PATH}")

In [ ]:
# (선택) GGUF 저장 — Ollama 로컬 실행용
GGUF_SAVE_PATH = "/content/drive/MyDrive/samseon/models/qwen3_5_gguf"

if False:  # True로 변경하면 실행
    os.makedirs(GGUF_SAVE_PATH, exist_ok=True)
    model.save_pretrained_gguf(GGUF_SAVE_PATH, quantization_method="q4_k_m")
    print(f"GGUF 저장 완료: {GGUF_SAVE_PATH}")